# AlphaZero Gomoku - Two-Phase Training

Notebook huấn luyện agent AlphaZero cho game Caro (Gomoku) 15x15.

**Thời gian ước tính:** 6-12 giờ trên Colab GPU (T4/V100)

## Quy trình 2 pha:
1. **Phase 1 - Học từ MiniMax:** Agent đấu với MiniMax ở các độ sâu tăng dần (1→3→5). Học từ nước đi "chuyên gia" của MiniMax → nhanh hơn nhiều so với self-play thuần.
2. **Phase 2 - Tự học (Self-play):** Sau khi đã mạnh, agent tự đấu với chính mình để vượt qua trình độ MiniMax.

## 1. Setup

In [ ]:
# Clone repo
!git clone https://github.com/VanLam05/Caro-AI.git
%cd Caro-AI

In [ ]:
# Install dependencies
!pip install torch numpy

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Cấu hình Training

### Hai pha training:
- **Phase 1 (vs MiniMax):** Agent chơi với MiniMax, học từ nước đi MiniMax. Độ khó MiniMax tăng dần: depth 1 → 2 → 3 → 5
- **Phase 2 (Self-play):** Agent tự chơi với chính mình để tiếp tục cải thiện

| Cấu hình | Phase 1 Iters | Phase 2 Iters | Games/Iter | Simulations | Thời gian |
|-----------|:---:|:---:|:---:|:---:|:---:|
| Nhanh     | 10  | 5   | 20  | 100 | ~3-4h  |
| Chuẩn     | 20  | 15  | 30  | 200 | ~8-12h |
| Kỹ        | 30  | 20  | 40  | 400 | ~18-24h|

In [ ]:
# Training configuration
CONFIG = {
    # Phase 1: Learn from MiniMax (faster learning)
    'phase1_iterations': 20,    # Iterations playing vs MiniMax
    'phase1_games': 30,         # Games per iteration vs MiniMax
    
    # Phase 2: Self-play refinement
    'phase2_iterations': 15,    # Iterations of self-play
    'phase2_games': 25,         # Games per self-play iteration
    
    # Common settings
    'num_simulations': 200,     # MCTS simulations per move
    'batch_size': 256,
    'lr': 0.002,
    'epochs_per_iter': 5,
    'num_res_blocks': 5,
    'channels': 128,
    'checkpoint_dir': 'neural_net',
}

## 3. Bắt đầu Training

Training sẽ chạy 2 pha tự động:
- Phase 1: Đấu với MiniMax (depth tăng dần 1→2→3→5)
- Phase 2: Self-play refinement

In [ ]:
import sys
sys.path.insert(0, '.')

from pipeline.train import AlphaZeroTrainer

trainer = AlphaZeroTrainer(
    phase1_iterations=CONFIG['phase1_iterations'],
    phase1_games=CONFIG['phase1_games'],
    phase2_iterations=CONFIG['phase2_iterations'],
    phase2_games=CONFIG['phase2_games'],
    num_simulations=CONFIG['num_simulations'],
    batch_size=CONFIG['batch_size'],
    lr=CONFIG['lr'],
    epochs_per_iter=CONFIG['epochs_per_iter'],
    num_res_blocks=CONFIG['num_res_blocks'],
    channels=CONFIG['channels'],
    checkpoint_dir=CONFIG['checkpoint_dir'],
)

print("Starting two-phase training...")
trainer.train()

## 4. Đánh giá kết quả

In [ ]:
# Load trained model and evaluate
from neural_net.architecture import GomokuNet
from mcts.mcts_alpha_zero import MCTS
from game.board import Board
from models.agentMiniMax import AgentMiniMax
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load best model
net = GomokuNet(board_size=15, in_channels=4, num_res_blocks=5, channels=128)
net.to(device)
net.load_checkpoint('neural_net/model_checkpoint.pth', device=device)
net.eval()

mcts = MCTS(net, num_simulations=200)

# Evaluate vs MiniMax at different depths
for depth in [1, 3, 5]:
    wins = 0
    num_eval_games = 20
    
    for g in range(num_eval_games):
        board = Board(rows=15, cols=15, winning_condition=5)
        minimax = AgentMiniMax(board, max_depth=depth)
        rl_player = 1 if g % 2 == 0 else -1
        
        move_count = 0
        while True:
            if board.turn == rl_player:
                action, _ = mcts.get_action(board, temperature=0)
                row, col = action
            else:
                move = minimax.get_move()
                if move is None:
                    break
                row, col = move
            
            board.make_move(row, col)
            move_count += 1
            
            result = board.get_game_ended()
            if result != 0:
                if result != 0.5:
                    if (result == 1 and rl_player == board.originXO) or \
                       (result == -1 and rl_player != board.originXO):
                        wins += 1
                break
            
            if move_count >= 225:
                break
    
    print(f"vs MiniMax depth={depth}: {wins}/{num_eval_games} wins ({wins/num_eval_games:.1%})")

## 5. Tải model về máy

Sau khi train xong, tải file `model_checkpoint.pth` về và đặt vào thư mục `neural_net/` trong project.

In [ ]:
# Download model file
from google.colab import files
files.download('neural_net/model_checkpoint.pth')

In [ ]:
# Alternatively, save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('neural_net/model_checkpoint.pth', '/content/drive/MyDrive/model_checkpoint.pth')
print("Model saved to Google Drive!")

## 6. Tiếp tục Training (Resume)

Nếu Colab bị ngắt, bạn có thể tiếp tục train từ checkpoint.

Dùng `--skip-phase1` để bỏ qua Phase 1 nếu đã hoàn thành.

In [ ]:
# Resume training from checkpoint
import os

# If you saved to Google Drive, copy back first:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('/content/drive/MyDrive/model_checkpoint.pth', 'neural_net/model_checkpoint.pth')

resume_trainer = AlphaZeroTrainer(
    phase1_iterations=0,   # Skip Phase 1 (already done)
    phase2_iterations=15,  # Continue with self-play
    phase2_games=25,
    num_simulations=200,
    checkpoint_dir='neural_net',
)

checkpoint_path = 'neural_net/model_checkpoint.pth'
if os.path.exists(checkpoint_path):
    resume_trainer.net.load_checkpoint(checkpoint_path, device=resume_trainer.device)
    print(f"Resumed from {checkpoint_path}")

resume_trainer.train()